# Global Jazz Diasporas
### Data preparation
This script creates geospatial data for Cisco Bradley's Global Jazz Diasporas project at the Pratt Institute Music & Migration Lab. It ingests the historical database of musician migrations and cleans for subsequent analysis. 

### Set up for replication
This script was designed to be run in ArcGIS Pro 3.5 using Python 3. To replicate, create a new ArcGIS project, add this notebook, and then run as follows. 

In [1]:
import arcpy
import os
import pandas

In [2]:
#set up workspaces and check database
aprx = arcpy.mp.ArcGISProject("Current")
default_gdb = aprx.defaultGeodatabase
default_folder = aprx.homeFolder
arcpy.env.overwriteOutput = True
print("Directory: " + default_folder)
print("Geodatabase: " + default_gdb)

Directory: C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Research\Music and Migration
Geodatabase: C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Research\Music and Migration\MusicMigration.gdb


### Load and edit tabular data
The source data is stored as a Google Sheet to enable ongoing updates by the research team. The script pulls a CSV from the Google Sheets API and then loads to a pandas data frame. It then processes the data for mapping by creating a unique identification key for each musician and each migration segment. Genre details are also defined and split into the wide data structure needed for subsequent mapping. 

In [ ]:
# load the data from Google Sheets
file_url = "contact research team"
data = pandas.read_csv(file_url)
data.columns

Index(['Unnamed: 0', 'Arrival State', 'Arrival County', 'Arrival City',
       'Musician Name', 'Surname', 'Native American', 'Arrival', 'Depart',
       'Depart Country', 'Depart State', 'Depart County', 'Depart City',
       'Final Destination', 'Genre', 'Instrument', 'Free', 'Dyn', 'Public',
       'Dynasty Name', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22'],
      dtype='object')

In [4]:
# drop unnessary columns
data = data.drop(columns = ['Unnamed: 0', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22'])
data.columns

Index(['Arrival State', 'Arrival County', 'Arrival City', 'Musician Name',
       'Surname', 'Native American', 'Arrival', 'Depart', 'Depart Country',
       'Depart State', 'Depart County', 'Depart City', 'Final Destination',
       'Genre', 'Instrument', 'Free', 'Dyn', 'Public', 'Dynasty Name'],
      dtype='object')

In [5]:
#create a unique ID for each musician 
data = data.sort_values(by='Musician Name')
data['MusicianID'] = data['Musician Name'].factorize()[0]+1

MusicianIDs = data['MusicianID'].nunique()
print(MusicianIDs, "musicians listed")

2509 musicians listed


In [6]:
#create a unique ID for each segement on the network
data['SegmentID'] = data['MusicianID'].astype(str) + data['Arrival'].astype(str) + data['Depart'].astype(str)

segmentIDs = data['SegmentID'].nunique()
print(segmentIDs, "unique travel paths listed")

15509 unique travel paths listed


In [10]:
# create unique IDs for origins and destinations
data['OriginID'] = data['Depart City'].astype(str) + '_' + data['MusicianID'].astype(str) + '_' + data['Depart'].astype(str)
data['DestinationID'] = data['Arrival City'].astype(str) + '_' + data['MusicianID'].astype(str) + '_' + data['Arrival'].astype(str)

originIDs = data['OriginID'].nunique()
destinationIDs = data['DestinationID'].nunique()

print(originIDs, "unique origins listed")
print(destinationIDs, "unique destinations listed")

13590 unique origins listed
14921 unique destinations listed


### Process genre data
The database includes an extensive directory of information about each musician, including the genre(s) they are most closely associated with. 

In [11]:
#classify genre information
## dictionary mapping abbreviations to genres
genre_mapping = {
    'AC': 'Afro-Cuban Jazz',
    'AF': 'African Jazz',
    'AJ': 'Acid Jazz',
    'AR': 'Acid Rock',
    'BB': 'Bebop',
    'BG': 'Boogaloo',
    'BL': 'Blues',
    'BN': 'Bossa Nova',
    'BR': 'Broadway',
    'BS': 'Brass Band',
    'BW': 'Boogie-Woogie',
    'CH': 'Spirituals',
    'CI': 'Circus',
    'CL': 'European Classical',
    'CV': 'Folk Cape Verde',
    'CY': 'Country',
    'DI': 'Disco',
    'DW': 'Doo-Wop',
    'EJ': 'Early Jazz',
    'EL': 'Easy Listening',
    'FJ': 'Free Jazz',
    'FK': 'Funk',
    'FO': 'Folk American',
    'FU': 'Fusion',
    'GJ': 'General Jazz',
    'GO': 'Gospel',
    'HB': 'Hard Bop',
    'HH': 'Hip Hop',
    'HO': 'House',
    'JB': 'Jump Blues',
    'JF': 'Jazz Funk',
    'LJ': 'Latin Jazz',
    'LT': 'Latin',
    'MB': 'Military Band',
    'MS': 'Minstrelsy',
    'MT': 'Motown',
    'MU': 'Musical Theater',
    'NW': 'New Wave',
    'OP': 'Opera',
    'PO': 'Pop',
    'RB': 'Rhythm and Blues',
    'RO': 'Rock',
    'RT': 'Ragtime',
    'SA': 'Salsa',
    'SJ': 'Soul Jazz',
    'SM': 'Smooth Jazz',
    'SO': 'Soul',
    'ST': 'Stride Piano',
    'SW': 'Swing',
    'VV': 'Vaudeville',
    'WF': 'World Fusion'
}

## ensure the data reads as text
data['Genre'] = data['Genre'].astype(str)

## spell out the acronyms into a new column
def spell_out(codes):
    # Split codes by spaces and semicolons
    codes_list = codes.replace(';', ' ').split()
    return ' '.join([genre_mapping[code] for code in codes_list if code in genre_mapping])
data['All Genres'] = data['Genre'].apply(spell_out)

## print the number of unique combinations
genre_combinations = data['All Genres'].nunique()
print(f"Unique genres: {len(genre_mapping)}")
print(f"Unique genre combinations: {genre_combinations}")

Unique genres: 51
Unique genre combinations: 329


In [12]:
# Create new variables to represent each genre
for code in genre_mapping.keys():
    data[code] = data['Genre'].apply(lambda x: int(code in x))

# verify the results
genre_sums = {code: data[code].sum() for code in genre_mapping.keys()}
print("Total records for each genre:")
print(genre_sums)

Total records for each genre:
{'AC': np.int64(7), 'AF': np.int64(4), 'AJ': np.int64(40), 'AR': np.int64(11), 'BB': np.int64(3376), 'BG': np.int64(2), 'BL': np.int64(5010), 'BN': np.int64(6), 'BR': np.int64(112), 'BS': np.int64(0), 'BW': np.int64(96), 'CH': np.int64(148), 'CI': np.int64(29), 'CL': np.int64(232), 'CV': np.int64(1), 'CY': np.int64(10), 'DI': np.int64(60), 'DW': np.int64(23), 'EJ': np.int64(2473), 'EL': np.int64(27), 'FJ': np.int64(2032), 'FK': np.int64(3), 'FO': np.int64(27), 'FU': np.int64(730), 'GJ': np.int64(962), 'GO': np.int64(483), 'HB': np.int64(1465), 'HH': np.int64(52), 'HO': np.int64(1), 'JB': np.int64(152), 'JF': np.int64(98), 'LJ': np.int64(5), 'LT': np.int64(2), 'MB': np.int64(171), 'MS': np.int64(33), 'MT': np.int64(1), 'MU': np.int64(13), 'NW': np.int64(15), 'OP': np.int64(78), 'PO': np.int64(362), 'RB': np.int64(1154), 'RO': np.int64(287), 'RT': np.int64(112), 'SA': np.int64(8), 'SJ': np.int64(450), 'SM': np.int64(113), 'SO': np.int64(461), 'ST': np.int64(

In [13]:
# rename the genre columns
data.rename(columns=genre_mapping, inplace=True)
data.columns

Index(['Arrival State', 'Arrival County', 'Arrival City', 'Musician Name',
       'Surname', 'Native American', 'Arrival', 'Depart', 'Depart Country',
       'Depart State', 'Depart County', 'Depart City', 'Final Destination',
       'Genre', 'Instrument', 'Free', 'Dyn', 'Public', 'Dynasty Name',
       'MusicianID', 'SegmentID', 'OriginID', 'DestinationID', 'All Genres',
       'Afro-Cuban Jazz', 'African Jazz', 'Acid Jazz', 'Acid Rock', 'Bebop',
       'Boogaloo', 'Blues', 'Bossa Nova', 'Broadway', 'Brass Band',
       'Boogie-Woogie', 'Spirituals', 'Circus', 'European Classical',
       'Folk Cape Verde', 'Country', 'Disco', 'Doo-Wop', 'Early Jazz',
       'Easy Listening', 'Free Jazz', 'Funk', 'Folk American', 'Fusion',
       'General Jazz', 'Gospel', 'Hard Bop', 'Hip Hop', 'House', 'Jump Blues',
       'Jazz Funk', 'Latin Jazz', 'Latin', 'Military Band', 'Minstrelsy',
       'Motown', 'Musical Theater', 'New Wave', 'Opera', 'Pop',
       'Rhythm and Blues', 'Rock', 'Ragtime', 'Sa

In [14]:
# sort and clean data
reorder = ['SegmentID', 'MusicianID', 'Musician Name', 'OriginID', 'DestinationID'] + [
    c for c in data.columns if c not in ['SegmentID', 'MusicianID', 'Musician Name', 'OriginID', 'DestinationID']]

data = data[reorder]

data = data.sort_values(by='SegmentID')
data

,SegmentID,MusicianID,Musician Name,OriginID,DestinationID,Arrival State,Arrival County,Arrival City,Surname,Native American,...,Rock,Ragtime,Salsa,Soul Jazz,Smooth Jazz,Soul,Stride Piano,Swing,Vaudeville,World Fusion
258,019621965,0,NaN,Middletown_0_1965,Paris_0_1962,Ile-de-France,Paris,Paris,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
3091,10001839nan,1000,"Harper, Elijah ""Buddy""",nan_1000_nan,Atlanta_1000_1839,Georgia,Fulton,Atlanta,Harper,NaN,...,0,0,0,0,0,0,0,0,0,0
14192,100018491895,1000,"Harper, Elijah ""Buddy""",Atlanta_1000_1895,nan_1000_1849,Virginia,NaN,NaN,Blackman,NaN,...,0,0,0,0,0,0,0,0,0,0
259,100116801700,1001,"Harrigan, Eric",Frederiksted_1001_1700,Marigot_1001_1680,St. Martin,Marigot,Marigot,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
424,100117001750,1001,"Harrigan, Eric",Spanish Town_1001_1750,Frederiksted_1001_1700,St. Croix,Frederiksted,Frederiksted,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
556,99818441889,998,"Harney, Richard ""Hacksaw""",Money_998_1889,nan_998_1844,Alabama,NaN,NaN,Howard,NaN,...,0,0,0,0,0,0,0,0,0,0
7428,99818891914,998,"Harney, Richard ""Hacksaw""",Greenville_998_1914,Money_998_1889,Mississippi,Leflore,Money,Harney,NaN,...,0,0,0,0,0,0,0,0,0,0
7905,99819141927,998,"Harney, Richard ""Hacksaw""",Memphis_998_1927,Greenville_998_1914,Mississippi,Washington,Greenville,Harney,NaN,...,0,0,0,0,0,0,0,0,0,0
13400,99919431961,999,"Harper, Billy",Denton_999_1961,Houston_999_1943,Texas,Harris,Houston,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0


In [15]:
# save as a CSV
data.to_csv(os.path.join(default_folder,"data_cleaned.csv"), index=False)
print(f"Saved to {default_folder}")

Saved to C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Research\Music and Migration
